In [7]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

# ------------------------------------------------------------
# Parameter
# ------------------------------------------------------------
H, L = 4.0, 28.0
cx, cy, R = 7.0, 0.0, 0.5

nu   = 1e-3
Umax = 0.3          # bewusst klein starten!
maxh = 0.4

maxit = 50
tol   = 1e-8

# ------------------------------------------------------------
# Geometrie + Mesh
# ------------------------------------------------------------
rect = MoveTo(0, -H/2).Rectangle(L, H).Face()
rect.edges.Min(X).name = "inlet"
rect.edges.Max(X).name = "outlet"
rect.edges.Min(Y).name = "walls"
rect.edges.Max(Y).name = "walls"

cyl = Circle((cx, cy), R).Face()
cyl.edges.name = "obstacle"

shape = rect - cyl
mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=maxh))
mesh.Curve(3)

# ------------------------------------------------------------
# FE-Räume (Taylor–Hood)
# ------------------------------------------------------------
V = VectorH1(mesh, order=2, dirichlet="inlet|walls|obstacle")
Q = H1(mesh, order=1, dirichlet="outlet")

X = FESpace([V, Q])
(u, p) = X.TrialFunction()
(v, q) = X.TestFunction()

# ------------------------------------------------------------
# Lösung & Startwert
# ------------------------------------------------------------
gfu = GridFunction(X)
gfu.vec[:] = 0.0

u_old = GridFunction(V)
u_old.vec[:] = 0.0

# Inlet-Profil
uin = CoefficientFunction((Umax*(1-(2*y/H)**2), 0))
gfu.components[0].Set(uin, definedon=mesh.Boundaries("inlet"))

# ------------------------------------------------------------
# LINEARER OPERATOR A(u_old)
# ------------------------------------------------------------
a = BilinearForm(X, symmetric=False)

# Viskosität
a += 2*nu*InnerProduct(Sym(Grad(u)), Sym(Grad(v))) * dx

# Druckkopplung
a += -div(v)*p * dx
a += -div(u)*q * dx

# Rechte Seite f = 0
F = LinearForm(X)
F.Assemble()

maxiter = 20

for i in range(maxiter):
    # Assemblieren von K und F, abhängig von der aktuellen Lösung gfu (für nichtlineare Probleme)
    a.Assemble()
    F.Assemble()

    # Residuum berechnen
    res = F.vec - a.mat * gfu.vec

    # Lösen des linearen Systems: K * du = res
    du = a.mat.Inverse(X.FreeDofs()) * res

    # Update der Lösung
    gfu.vec.data += du

    # Konvergenz prüfen
    if Norm(du) < tol:
        break

Draw(gfu.components[1], mesh, "p")
Draw(gfu.components[0], mesh, "u")

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene